In [1]:
import os
import random
import pickle
import math
import itertools
import time
import json
import datetime

from bidict import bidict
from datetime import datetime
import numpy as np
from scipy.spatial import distance
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from sklearn.manifold import TSNE
from sklearn import datasets, metrics

from farpoint_base import Auto_Oracle, COPK_Engine, NQ_Engine, Partition
from farpoint_testing import skew_weighted_accuracy

In [2]:
def printf(x):
    print(f'{x} = {eval(x)}')

In [3]:
"""
perf_metrics{
    'runs': [
        {'run_id': {
            'runtime' : <time to find all classes or to run out of queries. Whichever terminates the algorithm>,
            'class_v_query_dict' : <class_v_query_dict>
            'num_queries_to_find_all_classes' : <num_queries spent to find all classes>
           }
        }
    ],
    'max_num_queries' : <int num queries per run>,
    'num_runs' : <the number of runs contained in this structure (len('runs'))
        
}
"""

"\nperf_metrics{\n    'runs': [\n        {'run_id': {\n            'runtime' : <time to find all classes or to run out of queries. Whichever terminates the algorithm>,\n            'class_v_query_dict' : <class_v_query_dict>\n            'num_queries_to_find_all_classes' : <num_queries spent to find all classes>\n           }\n        }\n    ],\n    'max_num_queries' : <int num queries per run>,\n    'num_runs' : <the number of runs contained in this structure (len('runs'))\n        \n}\n"

In [4]:
ROOT='Server_Runs'

In [5]:
data_batches = []
for filename in os.listdir(ROOT):
    with open(os.path.join(ROOT,filename), 'r') as file:
        data_batches.append(json.load(file))

In [6]:
perf_metrics = data_batches[0]
for batch in data_batches[1:]:
    perf_metrics['runs']+=(batch['runs'])
    perf_metrics['num_runs']+=batch['num_runs']

In [7]:
total_queries = 0
total_runtime = 0
total_skwa = 0
overflow_count = 0
for i in range(perf_metrics['num_runs']):
    num_qs = perf_metrics['runs'][i]['num_queries_to_find_all_classes'] 
    runtime = perf_metrics['runs'][i]['runtime']
    skwa = perf_metrics['runs'][i]['accuracy_at_end']
    total_skwa+=skwa
    total_runtime += runtime
    if not np.isnan(num_qs):
        total_queries += num_qs
    else:
        overflow_count += 1

In [8]:
avg_queries = (total_queries) / (perf_metrics['num_runs']-overflow_count)
avg_runtime = total_runtime/(perf_metrics['num_runs'])
avg_skwa = total_skwa/(perf_metrics['num_runs'])
printf('avg_queries')
printf('avg_runtime')
printf('avg_skwa')

avg_queries = 163.3235294117647
avg_runtime = 22949.372934683888
avg_skwa = 0.7006755353293576


In [9]:
num_runs = perf_metrics['num_runs']
num_nans = overflow_count
printf('num_runs')
printf('num_nans')

num_runs = 55
num_nans = 21


In [10]:
perf_metrics['runs'][0]['accuracy_at_end']

0.7912267586838334